In [1]:
import os
import glob
import numpy as np
import math
from scipy.spatial import cKDTree
from tqdm import tqdm

DATASET_PATH = r"Novi Dataset\npy"
OUTPUT_PATH  = r"Novi Dataset\patches_8000_random"
TARGET       = 8000
NUM_CLASSES  = 4
SEED         = 42

os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f"Random patch rezanje | target: {TARGET} tocaka | izlaz: {OUTPUT_PATH} | seed: {SEED}")

Random patch rezanje | target: 8000 tocaka | izlaz: Novi Dataset\patches_8000_random | seed: 42


In [2]:
def random_patchevi(points, target=TARGET, seed=SEED):
    """
    Patch po patch: nasumicno izabere target tocaka od preostalih, makne ih, ponavlja.
    Zadnji (ako fali) dopunjen preklapanjem. Pokriva sve tocke.
    Seedano (random_state) pa je ponovljivo.
    Vraca listu nizova ORIGINALNIH indeksa (za rekonstrukciju).
    """
    N = points.shape[0]
    rng = np.random.default_rng(seed)
    tree_full = cKDTree(points)
    preostalo = np.ones(N, dtype=bool)
    patchevi = []

    while preostalo.sum() > 0:
        preostali_idx = np.where(preostalo)[0]

        if len(preostali_idx) >= target:
            # nasumicno target tocaka od preostalih (bez ponavljanja)
            patch = rng.choice(preostali_idx, size=target, replace=False)
            patchevi.append(patch)
            preostalo[patch] = False
        else:
            # manje od target -> uzmi sve + dopuni preklapanjem (najblizi vanjski)
            idx = preostali_idx
            fali = target - len(idx)
            centar = points[idx].mean(axis=0)
            k_trazi = min(N, len(idx) + fali * 3 + 10)
            _, kand = tree_full.query(centar, k=k_trazi)
            kand = np.atleast_1d(kand)
            u_patchu = set(idx.tolist())
            vanjski = [k for k in kand if k not in u_patchu][:fali]
            patch = np.concatenate([idx, np.array(vanjski, dtype=idx.dtype)])
            patchevi.append(patch)
            preostalo[idx] = False

    return patchevi

In [3]:
npy_pattern = os.path.join(DATASET_PATH, "tree_*", "*.npy")
all_npy_files = glob.glob(npy_pattern)
print(f"Found {len(all_npy_files)} files.")

Found 657 files.


In [4]:
total_patches = 0
patch_counts = []
preklapanja = []

for fp in tqdm(all_npy_files, desc="Random patch rezanje"):
    tree = os.path.basename(os.path.dirname(fp))
    cloud_name = os.path.splitext(os.path.basename(fp))[0]

    d = np.load(fp, allow_pickle=True).item()
    pts = np.array(d['points'].T, dtype=np.float32)     # (N, 3)
    cols = np.array(d['colors'].T, dtype=np.float32)    # (N, 3), 0-255
    lbls = np.array(d['labels'], dtype=np.int64)        # (N,)

    patchevi = random_patchevi(pts.astype(np.float64), TARGET, seed=SEED)

    out_dir = os.path.join(OUTPUT_PATH, tree, cloud_name)
    os.makedirs(out_dir, exist_ok=True)

    for i, orig_idx in enumerate(patchevi):
        out_fp = os.path.join(out_dir, f"{cloud_name}_{i}.npy")
        np.save(out_fp, {
            'points': pts[orig_idx],
            'colors': cols[orig_idx],
            'labels': lbls[orig_idx],
            'orig_idx': orig_idx.astype(np.int64),   # KLJUCNO za rekonstrukciju
            'tree_id': tree,
            'cloud_id': cloud_name,
            'patch_id': i,
            'n_original': pts.shape[0],
        })

    ukupno = sum(len(p) for p in patchevi)
    preklapanja.append(ukupno - pts.shape[0])
    total_patches += len(patchevi)
    patch_counts.append(len(patchevi))

patch_counts = np.array(patch_counts)
preklapanja = np.array(preklapanja)

print(f"\nGotovo. Ukupno patcheva: {total_patches}")
print(f"Patcheva po oblaku -> min: {patch_counts.min()} | max: {patch_counts.max()} | prosjek: {patch_counts.mean():.1f}")
print(f"Preklapanje -> prosjek: {preklapanja.mean():.0f} tocaka po oblaku")

Random patch rezanje: 100%|██████████| 657/657 [00:44<00:00, 14.81it/s]


Gotovo. Ukupno patcheva: 8107
Patcheva po oblaku -> min: 3 | max: 23 | prosjek: 12.3
Preklapanje -> prosjek: 4098 tocaka po oblaku


In [10]:
import numpy as np, glob, os

RANDOM_PATH = r"Novi Dataset\patches_8000_random"
tree = "tree_1_V_0000"
cloud = "0"

orig_fp = os.path.join(r"Novi Dataset\npy", tree, f"{cloud}.npy")
d = np.load(orig_fp, allow_pickle=True).item()
orig_pts = np.array(d['points'].T, dtype=np.float64)
N = orig_pts.shape[0]
print(f"ORIGINAL: {N} tocaka")

patch_dir = os.path.join(RANDOM_PATH, tree, cloud)
patch_files = glob.glob(os.path.join(patch_dir, "*.npy"))
print(f"Patcheva: {len(patch_files)}")

rek_pts = np.zeros((N, 3))
pokriveno = np.zeros(N, dtype=bool)
ukupno_patch_tocaka = 0
for fp in patch_files:
    p = np.load(fp, allow_pickle=True).item()
    oi = p['orig_idx']
    rek_pts[oi] = p['points']
    pokriveno[oi] = True
    ukupno_patch_tocaka += len(oi)

print(f"\nUkupno tocaka u svim patchevima (s preklapanjem): {ukupno_patch_tocaka}")
print(f"Jedinstvenih pokrivenih: {pokriveno.sum()} / {N}")
print(f"Sve pokriveno? {'DA' if pokriveno.all() else 'NE - fali ' + str(N - pokriveno.sum())}")
print(f"Koordinate identicne originalu? {np.allclose(rek_pts, orig_pts)}")

ORIGINAL: 128919 tocaka
Patcheva: 17

Ukupno tocaka u svim patchevima (s preklapanjem): 136000
Jedinstvenih pokrivenih: 128919 / 128919
Sve pokriveno? DA
Koordinate identicne originalu? True


In [ ]:
import numpy as np, glob, os
import open3d as o3d

RANDOM_PATH = r"Novi Dataset\patches_8000_random"
tree = "tree_1_V_0000"
cloud = "0"

orig_fp = os.path.join(r"Novi Dataset\npy", tree, f"{cloud}.npy")
d = np.load(orig_fp, allow_pickle=True).item()
points = np.array(d['points'].T, dtype=np.float64)
N = points.shape[0]
print(f"Oblak: {tree}/{cloud} | {N} tocaka")

patch_dir = os.path.join(RANDOM_PATH, tree, cloud)
patch_files = sorted(
    glob.glob(os.path.join(patch_dir, "*.npy")),
    key=lambda f: int(os.path.splitext(os.path.basename(f))[0].split("_")[-1])
)
print(f"Patcheva: {len(patch_files)} (svaki 8000 tocaka)")

patchevi_idx = []
for fp in patch_files:
    p = np.load(fp, allow_pickle=True).item()
    patchevi_idx.append(p['orig_idx'])

# --- 1) cijelo stablo, svaki patch svojom bojom ---
rng = np.random.default_rng(42)
boje = np.zeros((N, 3))
for oi in patchevi_idx:
    boje[oi] = rng.random(3)

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(boje)
o3d.visualization.draw_geometries([pcd],
    window_name=f"{tree}/{cloud} — {len(patch_files)} Random patcheva x 8000")

# --- 2) prvih 5 patcheva pojedinacno, SAMO te tocke ---
n_prikaz = min(5, len(patchevi_idx))
print(f"Prikazujem prvih {n_prikaz} patcheva (samo njihove tocke)")
for i in range(n_prikaz):
    oi = patchevi_idx[i]
    pts_p = points[oi]

    pcd_p = o3d.geometry.PointCloud()
    pcd_p.points = o3d.utility.Vector3dVector(pts_p)
    pcd_p.paint_uniform_color([1.0, 0.0, 0.0])

    o3d.visualization.draw_geometries([pcd_p],
        window_name=f"Patch {i} (redoslijed {i+1}.) | {len(oi)} tocaka")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Oblak: tree_1_V_0000/0 | 128919 tocaka
Patcheva: 17 (svaki 8000 tocaka)
Prikazujem prvih 5 patcheva (samo njihove tocke)


In [7]:
import numpy as np, glob, os
import open3d as o3d

RANDOM_PATH = r"Novi Dataset\patches_8000_random"
tree = "tree_1_V_0000"
cloud = "0"

orig_fp = os.path.join(r"Novi Dataset\npy", tree, f"{cloud}.npy")
d = np.load(orig_fp, allow_pickle=True).item()
points = np.array(d['points'].T, dtype=np.float64)
N = points.shape[0]
print(f"Oblak: {tree}/{cloud} | {N} tocaka")

patch_dir = os.path.join(RANDOM_PATH, tree, cloud)
patch_files = sorted(
    glob.glob(os.path.join(patch_dir, "*.npy")),
    key=lambda f: int(os.path.splitext(os.path.basename(f))[0].split("_")[-1])
)
print(f"Patcheva: {len(patch_files)} (svaki 8000 tocaka)")

patchevi_idx = []
for fp in patch_files:
    p = np.load(fp, allow_pickle=True).item()
    patchevi_idx.append(p['orig_idx'])

# --- zadnjih 5 patcheva pojedinacno, SAMO te tocke ---
n_ukupno = len(patchevi_idx)
n_prikaz = min(5, n_ukupno)
pocetak = n_ukupno - n_prikaz
print(f"Prikazujem zadnjih {n_prikaz} patcheva (redoslijed {pocetak}..{n_ukupno-1})")

for i in range(pocetak, n_ukupno):
    oi = patchevi_idx[i]
    pts_p = points[oi]

    pcd_p = o3d.geometry.PointCloud()
    pcd_p.points = o3d.utility.Vector3dVector(pts_p)
    pcd_p.paint_uniform_color([1.0, 0.0, 0.0])

    o3d.visualization.draw_geometries([pcd_p],
        window_name=f"Patch {i} (redoslijed {i+1}.) | {len(oi)} tocaka")

Oblak: tree_1_V_0000/0 | 128919 tocaka
Patcheva: 17 (svaki 8000 tocaka)
Prikazujem zadnjih 5 patcheva (redoslijed 12..16)
